# Projet B1 · L'assistant qui te fait réviser · ⭐⭐⭐

**Le problème** : relire ses notes donne l'impression de savoir, parce qu'on reconnaît le texte. Se faire interroger, non.
**Ce qu'on construit** : un assistant qui indexe **tes propres notes de cours** (RAG de la séance 11), retrouve les bons passages, **fabrique des questions de révision** et corrige tes réponses.
**Livrable** : ce notebook complété avec ton cours dans `MES_NOTES`, et sa fiche projet finale (taux de bonnes réponses, une question réussie, une ratée).

**Comment l'utiliser**
- Google Colab, rien à installer. Exécute chaque cellule avec `Maj + Entrée`.
- L'interrupteur `USE_MODEL` est en tête : `False` = mode démo, tout tourne **sans GPU et sans clé** (le faux modèle fabrique quand même des questions à partir des passages) ; `True` = le petit modèle Qwen tourne dans Colab. Le reste du notebook ne change pas.
- Les cellules **« À toi »** sont des exercices : elles s'exécutent telles quelles, la vérification affiche ✅ ou ❌, la solution est cachée juste en dessous — essaie avant de l'ouvrir.
- Ce projet est la suite directe de la [séance 11 · Le RAG](../../../seances/seance-11-rag/) : la cellule de préparation est la même.

## 0. Préparation

La cellule `llm(messages)` de la séance 11, à l'identique. Lance-la une fois.

In [ ]:
USE_MODEL = True   # ← mets False pour tester le notebook sans modèle (réponses factices, sans GPU)

import json, re
import numpy as np
import matplotlib.pyplot as plt

# ---------- Mode démo : un faux LLM qui répond sans réseau ni GPU ----------
def llm_factice(messages):
    """Mode démo : sans contexte, le faux modèle invente ; avec contexte, il recopie le passage le plus utile."""
    q = messages[-1]["content"]
    ql = q.lower()
    systeme = " ".join(m["content"] for m in messages if m["role"] == "system")
    if "Contexte :" in systeme:                                   # un RAG lui a donné des passages
        contexte = systeme.split("Contexte :", 1)[1]
        mots_question = {m for m in re.findall(r"\w{4,}", ql)} - {"quel", "quelle", "quels", "comment", "combien", "pourquoi", "dans", "avec", "pour"}
        phrases = [p.strip(" -\n") for p in re.split(r"(?<=[.!?])\s+", contexte) if p.strip(" -\n")]
        meilleure = max(phrases, key=lambda p: sum(m in p.lower() for m in mots_question), default="")
        if meilleure and sum(m in meilleure.lower() for m in mots_question) > 0:
            return "D'après tes documents : " + meilleure
        return "Je ne trouve pas cette information dans les documents fournis."
    if "sardine" in ql:
        return "Sardine Express est un jeu de cartes rapide où chaque joueur doit se débarrasser de ses sardines avant les autres. Il se joue avec 52 cartes classiques."
    if "prof" in ql or "cours" in ql:
        return "Le cours a lieu le lundi matin, dans la salle 12, avec le professeur Dupont."
    return "Bonne question ! Je ne suis pas certain, mais voici une réponse plausible : c'est un sujet intéressant."

# ---------- Le vrai modèle : petit modèle ouvert, gratuit, sans clé ----------

if USE_MODEL:
    %pip install -q transformers accelerate
    from transformers import pipeline
    _pipe = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", device_map="auto")

def llm(messages, max_new_tokens=150, temperature=0.7):
    """Envoie une liste de messages au modèle et renvoie sa réponse (du texte)."""
    if not USE_MODEL:
        return llm_factice(messages)
    if temperature == 0:      # température 0 = toujours la réponse la plus probable
        sortie = _pipe(messages, max_new_tokens=max_new_tokens, do_sample=False)
    else:
        sortie = _pipe(messages, max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature)
    return sortie[0]["generated_text"][-1]["content"].strip()

print("Modèle prêt :", "Qwen2.5-0.5B-Instruct" if USE_MODEL else "mode démo (llm_factice)")

**En option : le même appel avec une API.** Sur Colab, la clé est fournie par le formateur et rangée dans les **Secrets** (icône 🔑 à gauche), jamais dans le code. Décommente la cellule ci-dessous pour remplacer `llm()` par un gros modèle dans le cloud. Tout le reste du notebook ne change pas : il n'appelle que `llm(messages)`.

In [ ]:
# --- Variante API (à décommenter si le formateur a donné une clé) ---
# %pip install -q anthropic
# from google.colab import userdata          # les Secrets de Colab
# import anthropic
# client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))
#
# def llm(messages, max_new_tokens=150, temperature=0.7):
#     systeme = " ".join(m["content"] for m in messages if m["role"] == "system")
#     autres = [m for m in messages if m["role"] != "system"]
#     rep = client.messages.create(model="claude-opus-5", max_tokens=max_new_tokens,
#                                  system=systeme or "Tu es un assistant sympa.", messages=autres)
#     return rep.content[0].text.strip()
#
# Même idée avec Mistral : pip install mistralai, clé dans userdata.get("MISTRAL_API_KEY"),
# client.chat.complete(model="mistral-small-latest", messages=messages).choices[0].message.content

Les outils du projet : `demander()` (une question + un prompt système), les mots vides français, TF-IDF et le helper `verifier` qui affiche ✅ / ❌ sans jamais planter.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# mots trop fréquents pour aider la recherche ("le", "une", "quand"...) : TF-IDF les ignore
STOP_FR = ("le la les l un une des du de d et ou à a au aux en dans sur par pour avec sans ce cet cette ces se son sa ses "
           "leur leurs il elle ils elles on ne pas plus que qui quoi quel quelle quels dont où quand comment est sont être avoir fait y t").split()

REFUS = "Je ne trouve pas cette information dans mes notes."

def demander(question, systeme="Tu es un assistant sympa qui répond en français, en 2 phrases maximum."):
    return llm([{"role": "system", "content": systeme}, {"role": "user", "content": question}])

def verifier(nom, condition):
    """Affiche ✅ ou ❌ sans jamais planter. `condition` = un booléen, ou une fonction sans argument."""
    try:
        ok = bool(condition() if callable(condition) else condition)
    except Exception as e:
        ok = False
        print(f"   (erreur pendant la vérification : {type(e).__name__} : {e})")
    print(("✅ " if ok else "❌ ") + nom + ("" if ok else "  → pas encore, relis l'énoncé et réessaie"))

print("Outils prêts :", len(STOP_FR), "mots vides")

## 1. Contexte et question

Tu as un contrôle dans trois jours. Tu relis tes notes deux fois, tout te semble familier, et le jour du contrôle rien ne sort. C'est un piège connu : **reconnaître un texte n'est pas savoir le restituer**. Ce qui fait apprendre, c'est de sortir la réponse de sa tête — pas de la relire.

La question de ce projet est donc : **peut-on fabriquer automatiquement une interro à partir de n'importe quel cours ?** L'assistant doit poser des questions dont la réponse est vraiment dans le cours, indiquer le passage source de chaque question, et corriger la réponse donnée.

Un LLM seul ne sait pas le faire : il n'a jamais lu **tes** notes, donc il invente des questions plausibles mais hors programme. La solution est celle de la séance 11 : d'abord **chercher** les bons passages dans tes documents, ensuite seulement **faire écrire** le modèle, en lui interdisant de sortir du contexte fourni.

Le plan : découper tes notes → les indexer → chercher → générer des questions (question, réponse attendue, passage source) → passer l'interro → mesurer.

## 2. Les données : tes notes de cours

Les données de ce projet, ce sont **tes** notes. Pour que le notebook tourne dès la première exécution, un extrait de cours de SVT (quinze paragraphes) sert d'exemple. Tu le remplaceras par ton propre cours deux cellules plus bas.

Une règle de format, et une seule : **un paragraphe = une idée**, les paragraphes séparés par une ligne vide.

In [ ]:
COURS_SVT = """La cellule est la plus petite unité du vivant. Tous les êtres vivants, du champignon à l'éléphant, sont formés d'une seule cellule ou de milliards de cellules. Une cellule animale contient un noyau, du cytoplasme et une membrane qui la sépare de l'extérieur.

La cellule végétale possède en plus trois éléments : une paroi rigide faite de cellulose, une grande vacuole remplie d'eau, et des chloroplastes qui contiennent la chlorophylle. Ces trois éléments manquent dans la cellule animale.

La photosynthèse est la fabrication de matière organique par les végétaux chlorophylliens. À la lumière, la plante utilise de l'eau et du dioxyde de carbone pour produire du glucose, et elle rejette du dioxygène.

La chlorophylle est le pigment vert des chloroplastes. Elle capte l'énergie lumineuse du Soleil et permet la réaction de photosynthèse. Sans lumière, la photosynthèse s'arrête complètement.

Les plantes prélèvent l'eau et les sels minéraux dans le sol par les poils absorbants des racines. Ce mélange, appelé sève brute, monte ensuite jusqu'aux feuilles par des vaisseaux conducteurs.

La respiration cellulaire concerne toutes les cellules vivantes, végétales comme animales. Elles consomment du dioxygène et rejettent du dioxyde de carbone, de jour comme de nuit, pour libérer l'énergie contenue dans le glucose.

Il ne faut pas confondre respiration et photosynthèse. La respiration a lieu en permanence dans tous les êtres vivants, alors que la photosynthèse n'a lieu qu'à la lumière et seulement chez les végétaux chlorophylliens.

Chez l'être humain, les échanges gazeux se font dans les poumons, au niveau de minuscules sacs appelés alvéoles pulmonaires. Leur surface totale est immense, ce qui rend le passage du dioxygène vers le sang très rapide.

Le dioxygène est transporté dans le sang par l'hémoglobine, une molécule contenue dans les globules rouges. C'est elle qui donne au sang sa couleur rouge.

La digestion transforme les aliments en nutriments assez petits pour passer dans le sang. Ce sont les enzymes digestives, présentes dans la salive, l'estomac et l'intestin, qui découpent les aliments.

L'absorption des nutriments se fait dans l'intestin grêle, dont la paroi est couverte de villosités. Ces replis multiplient la surface de contact entre les nutriments et le sang.

Les besoins alimentaires se répartissent en glucides, lipides, protides, vitamines, sels minéraux et eau. Les glucides et les lipides servent surtout d'énergie, les protides servent surtout à construire et à réparer.

L'appareil circulatoire distribue les nutriments et le dioxygène à toutes les cellules. Le cœur est une pompe qui fonctionne sans arrêt : au repos, il bat environ soixante-dix fois par minute chez un adulte.

Dans le sol, les décomposeurs, c'est-à-dire les bactéries, les champignons et les vers de terre, transforment la matière organique morte en matière minérale. Cette matière minérale redevient utilisable par les plantes.

Une chaîne alimentaire commence toujours par un producteur, un végétal chlorophyllien qui fabrique sa matière organique. Viennent ensuite les consommateurs qui le mangent, puis les décomposeurs. Toute l'énergie de la chaîne vient au départ du Soleil."""

print(len(COURS_SVT.split("\n\n")), "paragraphes,", len(COURS_SVT.split()), "mots")
print(COURS_SVT[:220], "...")

À toi maintenant : colle **ton** cours entre les triples guillemets ci-dessous (au moins 10 paragraphes séparés par une ligne vide). Si tu laisses la variable vide, le notebook travaille sur le cours d'exemple — mais l'intérêt du projet, c'est ton cours à toi.

In [ ]:
MES_NOTES = """"""     # ← colle ton cours ici, entre les triples guillemets

CORPUS = MES_NOTES.strip() if MES_NOTES.strip() else COURS_SVT
print("Corpus utilisé :", "TES notes" if MES_NOTES.strip() else "le cours d'exemple (SVT)")
print(len(CORPUS.split()), "mots à indexer")

## 3. Découpage et indexation

Un RAG a cinq étapes : **découper, vectoriser, chercher, injecter, répondre**. Les deux premières se font une fois pour toutes, au début : c'est la construction de l'**index**.

### À toi · exercice 1 ⭐ · Découper le cours en chunks

Écris `decouper(texte)` : elle renvoie la liste des paragraphes (séparés par une ligne vide), sans espaces autour et sans paragraphe vide. Applique-la à `CORPUS` pour obtenir `chunks`.

Résultat attendu sur le cours d'exemple : 15 chunks, le premier commence par « La cellule ».

<details><summary>Indice</summary>

`texte.split("\n\n")` puis `.strip()` sur chaque morceau, en ne gardant que ceux qui ne sont pas vides.

</details>

<details><summary>Solution (à ouvrir après avoir essayé)</summary>

```python
def decouper(texte):
    """Un paragraphe (séparé par une ligne vide) = un chunk."""
    return [p.strip() for p in texte.split("\n\n") if p.strip()]

chunks = decouper(CORPUS)
print(len(chunks), "chunks")
for i, c in enumerate(chunks[:3]):
    print(f"[{i}] ({len(c.split())} mots) {c[:70]}...")
```

</details>

In [ ]:
# À toi
def decouper(texte):
    return None

chunks = decouper(CORPUS)
print(len(chunks) if chunks else None, "chunks")

In [ ]:
verifier("Exercice 1 · le découpage marche", lambda: len(decouper(COURS_SVT)) == 15)
verifier("Exercice 1 · pas d'espaces ni de vides", lambda: all(c == c.strip() and c for c in chunks))
verifier("Exercice 1 · chunks construits sur le corpus", lambda: len(chunks) == len(CORPUS.split("\n\n")))

# Filet de sécurité : le notebook continue même si l'exercice n'est pas fait.
if not chunks:
    def decouper(texte):
        return [p.strip() for p in texte.split("\n\n") if p.strip()]
    chunks = decouper(CORPUS)

In [ ]:
tailles = [len(c.split()) for c in chunks]
plt.figure(figsize=(7, 3))
plt.bar(range(len(tailles)), tailles)
plt.axhline(np.mean(tailles), color="red", linestyle="--", label=f"moyenne : {np.mean(tailles):.0f} mots")
plt.xlabel("numéro du chunk")
plt.ylabel("mots")
plt.title("Taille des chunks")
plt.legend()
plt.show()
print("le plus court :", min(tailles), "mots · le plus long :", max(tailles), "mots")

### À toi · exercice 2 ⭐⭐ · Construire l'index TF-IDF

Le graphique ci-dessus montre la taille des chunks : un chunk trop long noie la réponse dans du texte inutile, un chunk trop court perd le contexte. Entre 30 et 80 mots, on est bien.

On reste sur **TF-IDF** (scikit-learn) : il compte les mots importants, il ne comprend pas le sens, mais il tourne partout sans GPU et sans installation.

Entraîne un `TfidfVectorizer(stop_words=STOP_FR)` sur `chunks`, écris `vectoriser(textes)` qui renvoie un tableau numpy, et calcule `vecteurs` (un vecteur par chunk).

Résultat attendu : `vecteurs.shape` vaut (nombre de chunks, taille du vocabulaire).

<details><summary>Indice</summary>

`tfidf = TfidfVectorizer(stop_words=STOP_FR).fit(chunks)`, puis dans `vectoriser` : `tfidf.transform(textes).toarray()`.

</details>

<details><summary>Solution (à ouvrir après avoir essayé)</summary>

```python
tfidf = TfidfVectorizer(stop_words=STOP_FR).fit(chunks)   # le vocabulaire est appris sur NOS chunks

def vectoriser(textes):
    """Liste de textes → tableau numpy (une ligne = un vecteur)."""
    return tfidf.transform(textes).toarray()

vecteurs = vectoriser(chunks)
print("Forme du tableau :", vecteurs.shape, "→", len(chunks), "chunks,", vecteurs.shape[1], "mots de vocabulaire")
```

</details>

In [ ]:
# À toi
tfidf = None

def vectoriser(textes):
    """Liste de textes → tableau numpy (une ligne = un vecteur)."""
    return None

vecteurs = vectoriser(chunks)
print("Forme du tableau :", vecteurs.shape if vecteurs is not None else None)

In [ ]:
verifier("Exercice 2 · un vecteur par chunk", lambda: vecteurs.shape[0] == len(chunks))
verifier("Exercice 2 · un vocabulaire appris", lambda: vecteurs.shape[1] > 50)
verifier("Exercice 2 · vectoriser marche sur une question", lambda: vectoriser(["Que fait la chlorophylle ?"]).shape == (1, vecteurs.shape[1]))

# Filet de sécurité : le notebook continue même si l'exercice n'est pas fait.
if vecteurs is None:
    tfidf = TfidfVectorizer(stop_words=STOP_FR).fit(chunks)
    def vectoriser(textes):
        return tfidf.transform(textes).toarray()
    vecteurs = vectoriser(chunks)

**Option pour Colab avec GPU** : remplacer TF-IDF par des embeddings qui comprennent le sens.

```python
!pip install -q sentence-transformers
from sentence_transformers import SentenceTransformer
encodeur = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
def vectoriser(textes):
    return encodeur.encode(textes)
vecteurs = vectoriser(chunks)
```

Le reste du notebook ne change pas d'une ligne — c'est tout l'intérêt d'avoir isolé `vectoriser()`. Garde TF-IDF pour un premier passage : c'est plus rapide et ça rend les scores lisibles.

## 4. Recherche

Troisième étape : la question devient un vecteur à son tour, et on garde les `k` chunks les plus proches. C'est un moteur de recherche : il renvoie des **passages**, pas encore une réponse.

### À toi · exercice 3 ⭐⭐ · Chercher les k meilleurs passages

Écris `chercher(question, k=3)` qui renvoie une liste de `(chunk, score)`, triée du meilleur au moins bon, avec le score de similarité cosinus arrondi à 3 décimales.

Résultat attendu : pour « Que produit la photosynthèse ? », le premier passage parle de photosynthèse et son score dépasse 0,2.

<details><summary>Indice</summary>

`scores = cosine_similarity(vectoriser([question]), vecteurs)[0]`, puis `scores.argsort()[::-1][:k]` donne les indices des meilleurs.

</details>

<details><summary>Solution (à ouvrir après avoir essayé)</summary>

```python
def chercher(question, k=3):
    """Renvoie les k chunks les plus proches de la question, avec leur score."""
    scores = cosine_similarity(vectoriser([question]), vecteurs)[0]
    meilleurs = scores.argsort()[::-1][:k]
    return [(chunks[i], round(float(scores[i]), 3)) for i in meilleurs]

for passage, score in chercher("Que produit la photosynthèse ?"):
    print(f"[{score}] {passage[:80]}...")
```

</details>

In [ ]:
# À toi
def chercher(question, k=3):
    return None

for passage, score in chercher("Que produit la photosynthèse ?") or []:
    print(f"[{score}] {passage[:80]}...")

In [ ]:
resultats = chercher("Que produit la photosynthèse ?")
verifier("Exercice 3 · k résultats (chunk, score)", lambda: len(resultats) == 3 and len(resultats[0]) == 2)
verifier("Exercice 3 · triés du meilleur au moins bon", lambda: resultats[0][1] >= resultats[1][1] >= resultats[2][1])
verifier("Exercice 3 · le bon passage arrive en tête", lambda: "photosynthèse" in resultats[0][0].lower() and resultats[0][1] > 0.2)

# Filet de sécurité : le notebook continue même si l'exercice n'est pas fait.
if not resultats:
    def chercher(question, k=3):
        scores = cosine_similarity(vectoriser([question]), vecteurs)[0]
        return [(chunks[i], round(float(scores[i]), 3)) for i in scores.argsort()[::-1][:k]]

In [ ]:
for question in ["Qu'est-ce que l'hémoglobine ?", "Qui a écrit Les Misérables ?"]:
    scores = [s for _, s in chercher(question, k=3)]
    print(f"{question:35s} → meilleurs scores : {scores}")

### À toi · exercice 4 ⭐⭐ · Le seuil « je ne sais pas »

La cellule ci-dessus compare les scores d'une question **du cours** et d'une question **hors sujet** : c'est cet écart qui sert de garde-fou.

Écris `chercher_avec_seuil(question, k=3, seuil=SEUIL)` : comme `chercher`, mais elle ne garde que les passages dont le score atteint le seuil. Une question hors sujet doit donc renvoyer une liste **vide**.

Résultat attendu : liste vide pour « Qui a écrit Les Misérables ? », liste non vide pour « Qu'est-ce que l'hémoglobine ? ».

<details><summary>Indice</summary>

Une compréhension de liste sur le résultat de `chercher` : `[(p, s) for p, s in chercher(question, k) if s >= seuil]`.

</details>

<details><summary>Solution (à ouvrir après avoir essayé)</summary>

```python
SEUIL = 0.15

def chercher_avec_seuil(question, k=3, seuil=SEUIL):
    """Comme chercher, mais on jette les passages trop peu proches de la question."""
    return [(p, s) for p, s in chercher(question, k) if s >= seuil]

print("dans le cours :", len(chercher_avec_seuil("Qu'est-ce que l'hémoglobine ?")), "passages")
print("hors sujet    :", len(chercher_avec_seuil("Qui a écrit Les Misérables ?")), "passages")
```

</details>

In [ ]:
# À toi
SEUIL = 0.15

def chercher_avec_seuil(question, k=3, seuil=SEUIL):
    return None

print("dans le cours :", len(chercher_avec_seuil("Qu'est-ce que l'hémoglobine ?") or []), "passages")
print("hors sujet    :", len(chercher_avec_seuil("Qui a écrit Les Misérables ?") or []), "passages")

In [ ]:
verifier("Exercice 4 · hors sujet → rien", lambda: chercher_avec_seuil("Qui a écrit Les Misérables ?") == [])
verifier("Exercice 4 · question du cours → au moins 1 passage", lambda: len(chercher_avec_seuil("Qu'est-ce que l'hémoglobine ?")) >= 1)
verifier("Exercice 4 · le seuil est bien appliqué", lambda: all(s >= SEUIL for _, s in chercher_avec_seuil("Qu'est-ce que l'hémoglobine ?")))

# Filet de sécurité : le notebook continue même si l'exercice n'est pas fait.
if chercher_avec_seuil("Qu'est-ce que l'hémoglobine ?") is None:
    def chercher_avec_seuil(question, k=3, seuil=SEUIL):
        return [(p, s) for p, s in chercher(question, k) if s >= seuil]

## 5. Générer les questions de révision

C'est le cœur du projet. On ne demande pas au modèle « pose-moi des questions sur la SVT » — il inventerait des questions hors programme. On lui donne **un passage précis** et une consigne stricte : deux questions, leur réponse, et rien d'autre, uniquement à partir de ce passage.

Chaque question fabriquée garde donc trois choses : la **question**, la **réponse attendue**, et le **passage source** qui l'a produite. Ce troisième champ est ce qui permettra de corriger et de vérifier que rien n'a été inventé.

In [ ]:
CONSIGNE_QUESTIONS = """Tu es un professeur qui prépare une interro. À partir du passage de cours ci-dessous, écris 2 questions de révision et leur réponse.
Règles : questions courtes, en français, réponse UNIQUEMENT à partir du passage, aucune connaissance extérieure.
Format exact, une paire par bloc, rien d'autre :
Q: la question
R: la réponse
Contexte :
{passage}"""

print(CONSIGNE_QUESTIONS.format(passage=chunks[2])[:400], "...")

En mode démo (`USE_MODEL = False`), le faux modèle de la séance 11 ne sait que recopier un passage : il ne sait pas encore fabriquer des questions. On lui ajoute donc **une branche**, sans toucher à la cellule de préparation : si la consigne parle d'une interro, il fabrique une question par phrase du passage.

C'est du bricolage assumé — mais c'est ce qui permet de développer et de vérifier tout le reste du notebook sans GPU. Avec `USE_MODEL = True`, cette branche n'est jamais utilisée.

In [ ]:
_llm_factice_seance11 = llm_factice          # on garde la version de la leçon

def _questions_factices(passage):
    """Fabrique des paires Q/R plausibles à partir des phrases du passage (mode démo uniquement)."""
    phrases = [p.strip() for p in re.split(r"(?<=[.!?])\s+", passage) if len(p.split()) > 3]
    paires = []
    for phrase in phrases[:2]:
        mots = [m for m in re.findall(r"\w{5,}", phrase.lower()) if m not in STOP_FR][:3]
        sujet = " ".join(mots) if mots else "ce passage"
        paires.append(f"Q: Que dit le cours à propos de {sujet} ?\nR: {phrase}")
    return "\n".join(paires)

def llm_factice(messages):
    systeme = " ".join(m["content"] for m in messages if m["role"] == "system")
    if "prépare une interro" in systeme:                       # ---- mode générateur de questions ----
        return _questions_factices(systeme.split("Contexte :", 1)[1].strip())
    return _llm_factice_seance11(messages)                     # ---- tout le reste : séance 11 ----

print(llm_factice([{"role": "system", "content": CONSIGNE_QUESTIONS.format(passage=chunks[2])},
                   {"role": "user", "content": "Vas-y."}]))

### À toi · exercice 5 ⭐⭐ · Lire la sortie du modèle

Le modèle répond du texte : il faut le transformer en données utilisables. Écris `analyser_questions(texte, source="")` qui renvoie une liste de dictionnaires `{"question": ..., "reponse": ..., "source": ...}`.

Règles : une ligne qui commence par `Q:` ouvre une question, la ligne `R:` qui suit la referme. Une question sans réponse est ignorée. Les lignes en trop (« Voici les questions : ») sont ignorées aussi.

<details><summary>Indice</summary>

Parcours `texte.splitlines()`, garde une variable `courante`. `ligne.strip().startswith("Q:")` puis `ligne.split(":", 1)[1].strip()` pour le contenu.

</details>

<details><summary>Solution (à ouvrir après avoir essayé)</summary>

```python
def analyser_questions(texte, source=""):
    """Transforme la réponse du modèle (des lignes Q:/R:) en liste de dictionnaires."""
    questions, courante = [], None
    for ligne in texte.splitlines():
        ligne = ligne.strip()
        if ligne.upper().startswith("Q:"):
            courante = {"question": ligne.split(":", 1)[1].strip(), "reponse": "", "source": source}
        elif ligne.upper().startswith("R:") and courante:
            courante["reponse"] = ligne.split(":", 1)[1].strip()
            questions.append(courante)
            courante = None
    return questions

essai = """Voici les questions :
Q: Que capte la chlorophylle ?
R: L'énergie lumineuse du Soleil.
Q: Où se trouve la chlorophylle ?
R: Dans les chloroplastes.
Q: Une question sans réponse ?"""
print(analyser_questions(essai, source="passage 4"))
```

</details>

In [ ]:
# À toi
def analyser_questions(texte, source=""):
    return None

essai = """Voici les questions :
Q: Que capte la chlorophylle ?
R: L'énergie lumineuse du Soleil.
Q: Où se trouve la chlorophylle ?
R: Dans les chloroplastes.
Q: Une question sans réponse ?"""
print(analyser_questions(essai, source="passage 4"))

In [ ]:
lues = analyser_questions(essai, source="passage 4")
verifier("Exercice 5 · 2 paires complètes", lambda: len(lues) == 2)
verifier("Exercice 5 · les 3 champs sont remplis", lambda: lues[0]["question"] == "Que capte la chlorophylle ?" and lues[0]["reponse"] == "L'énergie lumineuse du Soleil." and lues[0]["source"] == "passage 4")
verifier("Exercice 5 · la question sans réponse est jetée", lambda: all("sans réponse" not in q["question"] for q in lues))

# Filet de sécurité : le notebook continue même si l'exercice n'est pas fait.
if not lues:
    def analyser_questions(texte, source=""):
        questions, courante = [], None
        for ligne in texte.splitlines():
            ligne = ligne.strip()
            if ligne.upper().startswith("Q:"):
                courante = {"question": ligne.split(":", 1)[1].strip(), "reponse": "", "source": source}
            elif ligne.upper().startswith("R:") and courante:
                courante["reponse"] = ligne.split(":", 1)[1].strip()
                questions.append(courante)
                courante = None
        return questions

### À toi · exercice 6 ⭐⭐⭐ · Fabriquer les questions d'un thème

Assemble tout : `fabriquer_questions(theme, k=3)` cherche les `k` meilleurs passages du thème, demande au modèle des questions **pour chaque passage** (avec `CONSIGNE_QUESTIONS`), analyse sa réponse et renvoie la liste complète.

Résultat attendu : au moins `k` questions, chacune avec sa `source` qui est bien un des chunks.

<details><summary>Indice</summary>

Boucle sur `chercher(theme, k)`. Pour chaque passage : `systeme = CONSIGNE_QUESTIONS.format(passage=passage)`, puis `demander("Écris les questions.", systeme=systeme)`, puis `analyser_questions(reponse, source=passage)` et `questions += ...`.

</details>

<details><summary>Solution (à ouvrir après avoir essayé)</summary>

```python
def fabriquer_questions(theme, k=3):
    """Cherche les k passages du thème, puis fait écrire des questions sur chacun."""
    questions = []
    for passage, score in chercher(theme, k):
        systeme = CONSIGNE_QUESTIONS.format(passage=passage)
        reponse = demander("Écris les questions.", systeme=systeme)
        questions += analyser_questions(reponse, source=passage)
    return questions

fiches = fabriquer_questions("la photosynthèse et la respiration", k=3)
print(len(fiches), "questions fabriquées")
```

</details>

In [ ]:
# À toi
def fabriquer_questions(theme, k=3):
    questions = []
    # pour chaque passage trouvé : construire la consigne, appeler le modèle, analyser la réponse
    return questions

fiches = fabriquer_questions("la photosynthèse et la respiration", k=3)
print(len(fiches), "questions fabriquées")

In [ ]:
verifier("Exercice 6 · au moins 3 questions", lambda: len(fiches) >= 3)
verifier("Exercice 6 · chaque question a ses 3 champs remplis", lambda: all(f["question"] and f["reponse"] and f["source"] for f in fiches))
verifier("Exercice 6 · la source est un vrai passage du cours", lambda: all(f["source"] in chunks for f in fiches))

# Filet de sécurité : le notebook continue même si l'exercice n'est pas fait.
if len(fiches) < 3:
    def fabriquer_questions(theme, k=3):
        questions = []
        for passage, score in chercher(theme, k):
            reponse = demander("Écris les questions.", systeme=CONSIGNE_QUESTIONS.format(passage=passage))
            questions += analyser_questions(reponse, source=passage)
        return questions
    fiches = fabriquer_questions("la photosynthèse et la respiration", k=3)

Voilà la fiche de révision. Regarde surtout la colonne « source » : chaque question est **attachée** à un passage du cours. Une question dont on ne saurait pas d'où elle vient serait exactement ce qu'on voulait éviter.

In [ ]:
for i, f in enumerate(fiches, 1):
    print(f"{i}. {f['question']}")
    print(f"   réponse attendue : {f['reponse']}")
    print(f"   source           : {f['source'][:70]}...\n")

## 6. Le mode « interro »

Maintenant on inverse les rôles : l'assistant pose, tu réponds, il corrige. La correction se fait **par recouvrement de mots-clés** entre ta réponse et la réponse attendue : c'est grossier, mais c'est mesurable, reproductible et honnête — et tu verras tout de suite ses limites, ce qui est le vrai sujet du projet [B4](../B4-fiabiliser-un-assistant/).

### À toi · exercice 7 ⭐⭐⭐ · Corriger une réponse

Écris `mots_cles(texte)` (les mots d'au moins 5 lettres, en minuscules, hors mots vides) puis `corriger(reponse_eleve, reponse_attendue)` qui renvoie la part des mots-clés attendus qu'on retrouve dans ta réponse, entre 0 et 1.

Résultat attendu : 1.0 si tu recopies la réponse attendue, 0.0 si tu ne réponds rien, et une valeur entre les deux sinon.

<details><summary>Indice</summary>

`set(re.findall(r"\w{5,}", texte.lower())) - set(STOP_FR)`. Puis `len(attendus & donnes) / len(attendus)`, en renvoyant 0.0 si `attendus` est vide.

</details>

<details><summary>Solution (à ouvrir après avoir essayé)</summary>

```python
def mots_cles(texte):
    """Les mots d'au moins 5 lettres, en minuscules, sans les mots vides."""
    return {m for m in re.findall(r"\w{5,}", texte.lower())} - set(STOP_FR)

def corriger(reponse_eleve, reponse_attendue):
    """Part des mots-clés attendus retrouvés dans la réponse de l'élève (0 à 1)."""
    attendus = mots_cles(reponse_attendue)
    if not attendus:
        return 0.0
    return len(attendus & mots_cles(reponse_eleve)) / len(attendus)

attendue = "Elle capte l'énergie lumineuse du Soleil et permet la photosynthèse."
print(corriger(attendue, attendue), corriger("", attendue), corriger("elle capte la lumière du soleil", attendue))
```

</details>

In [ ]:
# À toi
def mots_cles(texte):
    return set()

def corriger(reponse_eleve, reponse_attendue):
    return 0.0

attendue = "Elle capte l'énergie lumineuse du Soleil et permet la photosynthèse."
print(corriger(attendue, attendue), corriger("", attendue), corriger("elle capte la lumière du soleil", attendue))

In [ ]:
verifier("Exercice 7 · la réponse exacte vaut 1", lambda: corriger(attendue, attendue) == 1.0)
verifier("Exercice 7 · une réponse vide vaut 0", lambda: corriger("", attendue) == 0.0)
verifier("Exercice 7 · une réponse partielle est entre les deux", lambda: 0 < corriger("elle capte la lumière du soleil", attendue) < 1)
verifier("Exercice 7 · la casse et la ponctuation n'ont pas d'importance", lambda: corriger(attendue.upper() + " !!!", attendue) == 1.0)

# Filet de sécurité : le notebook continue même si l'exercice n'est pas fait.
if corriger(attendue, attendue) != 1.0:
    def mots_cles(texte):
        return {m for m in re.findall(r"\w{5,}", texte.lower())} - set(STOP_FR)
    def corriger(reponse_eleve, reponse_attendue):
        attendus = mots_cles(reponse_attendue)
        return len(attendus & mots_cles(reponse_eleve)) / len(attendus) if attendus else 0.0

L'interro : on pose les questions, on compare tes réponses aux réponses attendues, et on affiche le passage source quand c'est raté — c'est exactement ce qu'il faut relire.

Écris tes réponses dans `MES_REPONSES`, dans l'ordre des questions, puis relance la cellule suivante. La liste peut être plus courte que le nombre de questions : les questions sans réponse comptent zéro et affichent le passage à relire. (Pas d'`input()` : un notebook doit pouvoir se relancer de haut en bas sans qu'on tape quoi que ce soit.)

In [ ]:
MES_REPONSES = [                        # ← remplace ces exemples par TES réponses, une par question
    "Il ne faut pas confondre la respiration et la photosynthèse.",
    "La respiration a lieu en permanence chez tous les êtres vivants, la photosynthèse seulement à la lumière chez les végétaux chlorophylliens.",
    "",                                 # une réponse laissée vide : l'assistant te dira quoi relire
]

def interro(questions, reponses, seuil_de_reussite=0.4):
    """Pose les questions, corrige, et renvoie le score global."""
    scores = []
    for i, f in enumerate(questions):
        ma_reponse = reponses[i] if i < len(reponses) else ""
        score = corriger(ma_reponse, f["reponse"])
        scores.append(score)
        marque = "✅" if score >= seuil_de_reussite else "❌"
        print(f"{marque} Q{i + 1} · {f['question']}")
        print(f"     ta réponse       : {ma_reponse or '(rien)'}")
        print(f"     réponse attendue : {f['reponse']}")
        print(f"     score            : {score:.0%}")
        if score < seuil_de_reussite:
            print(f"     à relire         : {f['source'][:90]}...")
        print()
    moyenne = sum(scores) / len(scores) if scores else 0.0
    print(f"=== Score de l'interro : {moyenne:.0%} sur {len(scores)} questions ===")
    return moyenne

score_interro = interro(fiches, MES_REPONSES)

## 7. Évaluation

L'interro mesure **tes** connaissances. Reste à mesurer celles de **l'assistant** : quand on lui pose une question, retrouve-t-il la bonne information, et sait-il se taire quand la réponse n'est pas dans le cours ?

On écrit pour cela un petit banc de test : des questions dont la réponse est dans le cours (avec le mot qu'on doit voir apparaître), et des questions hors sujet où la seule bonne réponse est un refus.

### À toi · exercice 8 ⭐⭐⭐ · Répondre, ou refuser

Écris `repondre(question, k=3)` : elle cherche avec le seuil, **refuse** (`REFUS`) si aucun passage ne passe, sinon construit un prompt système avec les passages et appelle `demander`.

Le prompt système doit contenir la ligne `Contexte :` suivie des passages, et interdire de sortir du contexte.

<details><summary>Indice</summary>

`passages = chercher_avec_seuil(question, k)` ; si vide → `return REFUS`. Sinon `contexte = "\n".join(f"- {p}" for p, s in passages)` et un systeme qui finit par `f"Contexte :\n{contexte}"`.

</details>

<details><summary>Solution (à ouvrir après avoir essayé)</summary>

```python
def repondre(question, k=3):
    """RAG complet : chercher (avec seuil), injecter, répondre — ou refuser."""
    passages = chercher_avec_seuil(question, k)
    if not passages:
        return REFUS
    contexte = "\n".join(f"- {p}" for p, s in passages)
    systeme = ("Tu es l'assistant de révision de l'élève. Tu réponds en français, en 2 phrases maximum, "
               "UNIQUEMENT à partir du contexte ci-dessous. Si la réponse n'y est pas, dis : " + REFUS + "\n"
               f"Contexte :\n{contexte}")
    return demander(question, systeme=systeme)

print(repondre("Qu'est-ce qui transporte le dioxygène dans le sang ?"))
print(repondre("Qui a écrit Les Misérables ?"))
```

</details>

In [ ]:
# À toi
def repondre(question, k=3):
    return None

print(repondre("Qu'est-ce qui transporte le dioxygène dans le sang ?"))
print(repondre("Qui a écrit Les Misérables ?"))

In [ ]:
verifier("Exercice 8 · refus sur une question hors sujet", lambda: repondre("Qui a écrit Les Misérables ?") == REFUS)
verifier("Exercice 8 · la bonne information sort du cours", lambda: "hémoglobine" in repondre("Qu'est-ce qui transporte le dioxygène dans le sang ?").lower())
verifier("Exercice 8 · une vraie réponse, pas un refus", lambda: repondre("Que capte la chlorophylle ?") != REFUS)

# Filet de sécurité : le notebook continue même si l'exercice n'est pas fait.
if repondre("Qui a écrit Les Misérables ?") != REFUS:
    def repondre(question, k=3):
        passages = chercher_avec_seuil(question, k)
        if not passages:
            return REFUS
        contexte = "\n".join(f"- {p}" for p, s in passages)
        systeme = ("Tu es l'assistant de révision de l'élève. Tu réponds en français, en 2 phrases maximum, "
                   "UNIQUEMENT à partir du contexte ci-dessous. Si la réponse n'y est pas, dis : " + REFUS + "\n"
                   f"Contexte :\n{contexte}")
        return demander(question, systeme=systeme)

Le banc de test : six questions du cours (avec le mot-clé attendu) et deux questions hors sujet (`None` = un refus est attendu). Si tu travailles sur **tes** notes, remplace ces huit lignes par les tiennes — c'est l'exercice le plus utile de la section.

In [ ]:
BANC = [
    {"question": "Quel gaz la plante rejette-t-elle grâce à la photosynthèse ?", "attendu": "dioxygène"},
    {"question": "Que contiennent les chloroplastes ?", "attendu": "chlorophylle"},
    {"question": "Où se font les échanges gazeux dans les poumons ?", "attendu": "alvéoles"},
    {"question": "Qu'est-ce qui transporte le dioxygène dans le sang ?", "attendu": "hémoglobine"},
    {"question": "En quoi la digestion transforme-t-elle les aliments ?", "attendu": "nutriments"},
    {"question": "Que trouve-t-on dans une cellule végétale et pas dans une cellule animale ?", "attendu": "cellulose"},
    {"question": "Qui a écrit Les Misérables ?", "attendu": None},
    {"question": "Quel est le prix d'un billet de train ?", "attendu": None},
]
print(len(BANC), "questions dont", sum(1 for b in BANC if b["attendu"] is None), "hors sujet")

In [ ]:
def evaluer(banc):
    """Passe le banc de test et renvoie (taux de réussite, détail ligne par ligne)."""
    detail = []
    for cas in banc:
        reponse = repondre(cas["question"])
        if cas["attendu"] is None:
            ok = reponse == REFUS
        else:
            ok = cas["attendu"].lower() in reponse.lower()
        detail.append({"question": cas["question"], "hors_sujet": cas["attendu"] is None,
                       "ok": ok, "reponse": reponse})
    taux = sum(d["ok"] for d in detail) / len(detail)
    return taux, detail

taux, detail = evaluer(BANC)
for d in detail:
    print(("✅" if d["ok"] else "❌"), ("[hors sujet] " if d["hors_sujet"] else "") + d["question"])
    print("    →", d["reponse"][:110])
print(f"\n=== Taux de bonnes réponses : {taux:.0%} ({sum(d['ok'] for d in detail)}/{len(detail)}) ===")

In [ ]:
sur_le_cours = [d for d in detail if not d["hors_sujet"]]
hors_sujet = [d for d in detail if d["hors_sujet"]]
valeurs = [sum(d["ok"] for d in sur_le_cours) / len(sur_le_cours), sum(d["ok"] for d in hors_sujet) / len(hors_sujet)]

plt.figure(figsize=(5, 3.5))
plt.bar(["questions du cours", "questions hors sujet"], valeurs, color=["tab:green", "tab:orange"])
plt.ylim(0, 1.05)
plt.ylabel("part de bonnes réponses")
plt.title(f"Mon assistant : {taux:.0%} sur {len(detail)} questions")
for i, v in enumerate(valeurs):
    plt.text(i, v + 0.02, f"{v:.0%}", ha="center")
plt.show()

**À regarder de près** : une question au moins devrait échouer, et c'est instructif. Sur le cours d'exemple, « Que trouve-t-on dans une cellule végétale **et pas** dans une cellule animale ? » ramène le passage sur la cellule animale : TF-IDF compte les mots, il ne voit pas la négation. Regarde laquelle échoue chez toi et dis pourquoi — c'est la ligne la plus intéressante de ton rapport.

Ensuite, les deux barres ne mesurent pas la même chose. À gauche, l'assistant *retrouve* l'information ; à droite, il *renonce*. Un assistant qui refuse tout aurait 100 % à droite et 0 % à gauche — et serait inutile. C'est le réglage du `SEUIL` qui déplace le curseur entre les deux.

Essaie : remets `SEUIL = 0.05` puis `SEUIL = 0.30` dans la cellule de l'exercice 4, relance-la ainsi que l'évaluation, et note les deux taux. Tu viens de faire, à la main, ce que le projet [B4](../B4-fiabiliser-un-assistant/) automatise.

## 8. Conclusion et fiche projet

À retenir :

- **Chercher d'abord, faire écrire ensuite.** Le modèle ne fabrique de bonnes questions que si on lui donne un passage précis et l'interdiction d'en sortir.
- **Chaque question garde sa source.** C'est ce qui permet de corriger, de relire le bon paragraphe, et de vérifier que rien n'a été inventé.
- **Un assistant utile sait dire non.** Le seuil de similarité est un garde-fou qui coûte quelques bonnes réponses et évite beaucoup d'inventions.
- **TF-IDF cherche des mots, pas du sens.** Une question reformulée avec d'autres mots peut échouer : c'est la limite à connaître, et l'argument pour passer aux embeddings.

Remplis la fiche ci-dessous : c'est ce que tu montreras.

In [ ]:
MA_SYNTHESE = """(à remplir) Mon assistant révise le cours de ... . Il a bien posé la question ... ,
et il a raté ... parce que ... . Le seuil que j'ai gardé est ... , parce que ... ."""

print("=== FICHE PROJET B1 · L'ASSISTANT QUI TE FAIT RÉVISER ===")
print(f"Corpus            : {'mes notes' if MES_NOTES.strip() else 'cours d exemple (SVT)'} · {len(chunks)} chunks · {len(CORPUS.split())} mots")
print(f"Index             : TF-IDF, {vecteurs.shape[1]} mots de vocabulaire · seuil = {SEUIL}")
print(f"Questions générées: {len(fiches)}")
print(f"Score de l'interro: {score_interro:.0%}")
print(f"Assistant évalué  : {taux:.0%} sur {len(BANC)} questions ({sum(d['ok'] for d in sur_le_cours)}/{len(sur_le_cours)} sur le cours, {sum(d['ok'] for d in hors_sujet)}/{len(hors_sujet)} refus corrects)")
print(f"Mode              : {'modèle Qwen2.5-0.5B' if USE_MODEL else 'démo (llm_factice)'}")
print("\nMa synthèse :", MA_SYNTHESE)

## Pour aller plus loin

- **Passe aux embeddings** (`sentence-transformers`, cellule optionnelle de la section 3) et compare les deux méthodes sur le même banc de test : même tableau, deux colonnes. C'est le graphique qui fait la meilleure diapositive.
- **Ajoute la révision espacée** : les questions ratées reviennent à l'interro suivante, celles réussies deux fois sortent du paquet. Il suffit d'un dictionnaire `{question: nombre de réussites}` sauvegardé en JSON.
- **Fiabilise-le** : le projet [B4](../B4-fiabiliser-un-assistant/) reprend cet assistant, lui construit un banc de 15 questions, mesure ses hallucinations et lui pose trois garde-fous.

Liens utiles : le modèle https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct · TF-IDF https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html · similarité cosinus https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html · embeddings multilingues https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 · un RAG pas à pas https://www.kaggle.com/code/markishere/day-2-document-q-a-with-rag · le même service prêt à l'emploi https://notebooklm.google